# 04 · Measure one idle connection cycle

Time `connect()` (port open plus model/serial/firmware queries) and `close()` separately. Release is attempted even if identification fails. No status polling or control setting is included, and no equipment stop is sent.

Read [the tutorial guide](README.md) and [physical commissioning](../docs/commissioning.md) first. Use one serial owner. These notebook outputs are intentionally empty; no bench result is implied.


## Environment

From the repository root, run:

```powershell
uv sync --locked
uv run --locked --with jupyterlab --with ipykernel python -m jupyterlab
```

Choose that environment's Python kernel and run top to bottom. Restart the kernel after driver changes. The next cell opens no serial port.


In [ ]:
import sys
from pathlib import Path

# Prefer this checkout when launched from the repository root or examples/.
REPO_ROOT = Path.cwd().resolve()
for candidate in (REPO_ROOT, REPO_ROOT.parent):
    if (candidate / "src/tark_nrc/controller.py").is_file():
        REPO_ROOT = candidate
        sys.path.insert(0, str(candidate / "src"))
        break
print(f"Python: {sys.executable}")
print(f"Repository/config base: {REPO_ROOT}")

## Imports


In [ ]:
from pathlib import Path
from time import perf_counter

from tark_nrc import create_chiller

## Choose connection settings

Set `CONFIG_PATH` to your verified NRC400 profile. This keeps USB adapter/unit identity bindings and verified serial framing/timeouts together. The current `[control]` section accepts only `expected_serial_number`; remove old `allow_writes`, `min_c` and `max_c` entries. The API has no write-enable switch, target range or state-based control lock. Choose targets for the actual coolant and experiment. There is no guessed COM port. `SIMULATE = True` ignores the hardware profile and opens no physical port; each run creates a fresh simulated device with no shared state. Review all operation settings before continuing.


In [ ]:
CONFIG_PATH = REPO_ROOT / "config.toml"
SIMULATE = False

## Time connection and release, including on connection failure

Run this entire operation cell together: connection and cleanup share one context/finally block. Ctrl+C/Interrupt attempts cleanup while allowing Python to finish. Do not split the cell, kill the kernel during I/O, or rerun setup over an unfinished connection. Errors remain visible. Closing the port retains the equipment's target, loop and pump state.


In [ ]:
chiller = create_chiller(None if SIMULATE else CONFIG_PATH, simulate=SIMULATE)
print(f"Backend: {chiller.transport.backend}; timeout: {chiller.transport.settings.timeout_s:g} s")
connect_ok = close_ok = False
try:
    started = perf_counter()
    try:
        chiller.connect()  # Opens the selected port and queries MOD, SSN and VER.
        connect_ok = True
    finally:
        print(f"connect(): {(perf_counter() - started) * 1000:.3f} ms | success={connect_ok}")
    print(f"Identity: {chiller.identity}")
finally:
    started = perf_counter()
    try:
        chiller.close()  # Attempt release even if connection/identification failed.
        close_ok = True
    finally:
        print(f"close():   {(perf_counter() - started) * 1000:.3f} ms | success={close_ok}")

print("Idle cycle complete. This does not measure shutdown during active I/O.")
print("Connection closure does not stop equipment.")

## Check results and finish

Retain the success flags and durations with the unit/adapter/profile. This one idle cycle is not active-I/O shutdown timing or a physical stop latency test.

**Software connection closure does not stop the chiller or pump.** Follow the physical shutdown procedure separately and confirm panel state. A forced kernel termination cannot guarantee cleanup. See [the hardware test record checklist](README.md#6-record-a-hardware-test-result) for evidence to retain.
